# benchmark_hardware_analysis

Interprets results from the C++ `benchmark_hardware` binary.  
Canonical data store: `results/benchmark_hardware/<model>/<config>_<scenario>[_dpuN][_entN].json`

Sections
- §0 Setup & data load
- §1 Inventory
- §2 Stage breakdown
- §3 S0 → S1 speedup
- §4 Throughput scaling (nn_only / entropy_only)
- §5 Roofline
- §6 Power & energy
- §7 Triple tradeoff (bpp × PSNR × HW perf)

## §0 Setup

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd

ROOT_DIR = Path("..").resolve()
BENCH_DIR = ROOT_DIR / "results" / "benchmark_hardware"
COMP_DIR = ROOT_DIR / "results" / "fpga" / "compiled_models"
PLOTS_DIR = ROOT_DIR / "results" / "plots" / "benchmark_hardware"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

# Add notebooks/ to path so _benchmark_loader is importable
sys.path.insert(0, str(ROOT_DIR / "notebooks"))
from _benchmark_loader import (
    load_runs,
    load_stage_breakdowns,
    load_quality_metrics,
    join_hw_quality,
    IDENTITY_COLS,
)

# Shared colour palette
C = json.load(open(ROOT_DIR / "notebooks" / "plots_colors.json"))
ARCH_CLR = C["architectures"]
CFG_CLR = C["bench_configs"]
SCEN_CLR = C["bench_scenarios"]
STAGE_CLR = C["cpp_stages"]

# Architecture display names (paper notation) and canonical plot order (fast → slow)
ARCH_LABEL = {"FP": "FP", "SHyp": "SH", "ResFP": "ResFP", "ResSHyp": "ResSH"}
ARCH_ORDER = ["FP", "SHyp", "ResFP", "ResSHyp"]

# ANSI colour shortcuts for print statements
r = "\033[31m"
y = "\033[33m"
g = "\033[32m"
b = "\033[34m"
e = "\033[0m"

SAVE_FIGURES = True
FIGURE_FORMAT = "pdf"
FIG_DPI = 120


def savefig(name: str) -> None:
    if SAVE_FIGURES:
        path = PLOTS_DIR / f"{name}.{FIGURE_FORMAT}"
        plt.savefig(path, bbox_inches="tight", dpi=FIG_DPI)
        print(f"{g}Saved → {path}{e}")


print(f"{b}ROOT_DIR ={e} {ROOT_DIR}")
print(f"{b}BENCH_DIR={e} {BENCH_DIR}")

In [ ]:
runs_df = load_runs(BENCH_DIR)
stages_df = load_stage_breakdowns(BENCH_DIR)
quality_df = load_quality_metrics(COMP_DIR)
joined_df = join_hw_quality(runs_df, quality_df)

print(f"{b}runs_df:   {e}{len(runs_df)} rows")
print(f"{b}stages_df: {e}{len(stages_df)} rows")
print(f"{b}quality_df:{e}{len(quality_df)} rows")

## §1 Inventory

In [ ]:
display(
    runs_df[
        IDENTITY_COLS
        + [
            "iters",
            "warmup",
            "subset_patches",
            "evaluated_at",
            "throughput_fps",
            "total_latency_mean_ms",
        ]
    ].to_string()
)

In [ ]:
# Coverage grid: which (arch × config × scenario) cells are populated?
CONFIGS = ["s0", "s1", "nn_only", "entropy_only"]
SCENARIOS = ["compress", "full"]
ARCHS = sorted(runs_df["arch"].unique())

print(f"{b}Coverage grid (✓ = at least one run; — = missing):{e}")
for arch in ARCHS:
    print(f"\n  {b}{arch}{e}")
    subset = runs_df[runs_df["arch"] == arch]
    for cfg in CONFIGS:
        for scen in SCENARIOS:
            has = ((subset["config"] == cfg) & (subset["scenario"] == scen)).any()
            # ceiling configs carry their parallelism in dpu_cores / entropy_threads
            if cfg in ("nn_only", "entropy_only") and scen == "full":
                continue  # not applicable
            sym = f"{g}✓{e}" if has else f"{r}—{e}"
            print(f"    {cfg:15s} / {scen:8s}: {sym}")

# Ceiling runs (dpu_cores > 1 or entropy_threads > 1)
ceiling_runs = runs_df[(runs_df["dpu_cores"] > 1) | (runs_df["entropy_threads"] > 1)]
if not ceiling_runs.empty:
    print(f"\n{b}Ceiling runs:{e}")
    print(ceiling_runs[IDENTITY_COLS + ["throughput_fps"]].to_string(index=False))

## §2 Stage breakdown

Stacked horizontal bar: per (model × scenario), how much of the total latency each stage consumes.  
Only S0 runs are used here (S1 timers overlap; s0 gives the true sequential breakdown).

In [ ]:
import re as _re
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

# Canonical temporal execution order of stages (covers all arch × scenario combos)
STAGE_ORDER = [
    "normalize",
    "g_a",
    "h_a",
    "eb_compress",
    "eb_decompress",
    "h_s",
    "gc_compress",
    "gc_decompress",
    "g_s",
    "denorm",
]
_STAGE_RANK = {s: i for i, s in enumerate(STAGE_ORDER)}


def stacked_bar_scenario(scenario: str, ax: plt.Axes, ylim: float, config: str = "s0") -> None:
    """Vertical stacked bar: one bar per arch (in ARCH_ORDER), stages in temporal order."""
    if config != "s0" and config != "s1":
        raise ValueError("Currently only supports config 's0' or 's1'")
    # One representative model per arch (L1000 preferred)
    arch_to_model = {}
    for arch in runs_df["arch"].unique():
        candidates = runs_df[
            (runs_df["arch"] == arch)
            & (runs_df["config"] == config)
            & (runs_df["scenario"] == scenario)
        ]["model_name"].tolist()
        if not candidates:
            continue
        pref = [m for m in candidates if "L1000" in m]
        arch_to_model[arch] = pref[0] if pref else candidates[0]

    # Respect canonical arch order (fast → slow); skip archs with no data
    archs = [a for a in ARCH_ORDER if a in arch_to_model]

    for xi, arch in enumerate(archs):
        model_name = arch_to_model[arch]
        sel = stages_df[
            (stages_df["model_name"] == model_name)
            & (stages_df["config"] == config)
            & (stages_df["scenario"] == scenario)
        ].copy()
        if sel.empty:
            continue
        sel["_rank"] = sel["stage"].map(lambda s: _STAGE_RANK.get(s, len(STAGE_ORDER)))
        sel = sel.sort_values("_rank").drop(columns=["_rank"])
        total = sel["mean_ms"].sum()
        bottom = 0.0
        for _, row in sel.iterrows():
            h = row["mean_ms"]
            pct = 100 * h / total
            ax.bar(
                xi,
                h,
                bottom=bottom,
                width=0.6,
                color=STAGE_CLR.get(row["stage"], "#888888"),
                edgecolor="white",
                linewidth=0.3,
            )
            if pct > 5:
                ax.text(
                    xi,
                    bottom + h / 2,
                    f"{pct:.0f}%",
                    ha="center",
                    va="center",
                    fontsize=8,
                    color="white",
                    fontweight="bold",
                )
            bottom += h
        ax.text(xi, total + ylim * 0.015, f"{total:.1f} ms", ha="center", va="bottom", fontsize=8)

    ax.set_xticks(range(len(archs)))
    ax.set_xticklabels([ARCH_LABEL.get(a, a) for a in archs], fontsize=10)
    ax.set_ylabel("latency (ms)")
    ax.set_ylim(0, ylim)
    ax.set_title(f"Stage latency breakdown ({config}) — {scenario}")

    # Legend: reversed so top-of-bar stage (last temporal) appears first in list
    present = set(stages_df[stages_df["scenario"] == scenario]["stage"])
    handles = [
        Patch(facecolor=STAGE_CLR.get(s, "#888888"), label=s)
        for s in reversed(STAGE_ORDER)
        if s in present
    ]
    ax.legend(handles=handles, loc="upper left", fontsize=8, title="stage")


# fig, ax = plt.subplots(figsize=(9, 6), dpi=FIG_DPI)
# stacked_bar_scenario("compress", ax, ylim=100, config="s0")
# fig.tight_layout()
# savefig("stage_breakdown_compress_s0")
# plt.show()

fig, ax = plt.subplots(figsize=(9, 6), dpi=FIG_DPI)
stacked_bar_scenario("compress", ax, ylim=100, config="s1")
fig.tight_layout()
savefig("stage_breakdown_compress_s1")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6), dpi=FIG_DPI)
stacked_bar_scenario("full", ax, ylim=200, config="s0")
fig.tight_layout()
savefig("stage_breakdown_full_s0")
plt.show()

## §3 S0 → S1 speedup

For each architecture, g_a and g_s wall-clock speedup when switching from S0 (sequential) to S1 (channel-parallel).  
Larger residual networks (ResSHyp) benefit more because longer DPU inference amortises the thread-launch overhead better.

In [ ]:
from matplotlib.patches import Patch


def s1_speedup(model_name: str, scenario: str) -> dict:
    """Return per-stage ms for s0 and s1, plus derived speedup."""

    def stage_ms(cfg, stage):
        row = stages_df[
            (stages_df["model_name"] == model_name)
            & (stages_df["config"] == cfg)
            & (stages_df["scenario"] == scenario)
            & (stages_df["stage"] == stage)
        ]
        return float(row["mean_ms"].iloc[0]) if not row.empty else float("nan")

    result = {}
    for stage in ("g_a", "g_s"):
        s0_ms = stage_ms("s0", stage)
        s1_ms = stage_ms("s1", stage)
        result[stage] = {
            "s0_ms": s0_ms,
            "s1_ms": s1_ms,
            "speedup": s0_ms / s1_ms if s1_ms and s1_ms > 0 else float("nan"),
        }
    return result


# Gather speedup data
speedup_rows = []
for model in sorted(runs_df["model_name"].unique()):
    arch = runs_df[runs_df["model_name"] == model]["arch"].iloc[0]
    for scen in ["compress", "full"]:
        spd = s1_speedup(model, scen)
        for stage in ("g_a", "g_s"):
            if not any(np.isnan(v) for v in spd[stage].values()):
                speedup_rows.append(
                    {
                        "model_name": model,
                        "arch": arch,
                        "scenario": scen,
                        "stage": stage,
                        **spd[stage],
                    }
                )

spd_df = pd.DataFrame(speedup_rows)

if spd_df.empty:
    print(f"{y}No S0/S1 pairs found — run both configs to see speedup plot.{e}")
else:
    fig, axes = plt.subplots(1, 2, figsize=(13, 6), dpi=FIG_DPI)
    w = 0.35
    arch_order_map = {a: i for i, a in enumerate(ARCH_ORDER)}
    stage_order_map = {"g_a": 0, "g_s": 1}

    for ax, scen in zip(axes, ["compress", "full"]):
        sub = spd_df[spd_df["scenario"] == scen].copy()
        sub["_arch_rank"] = sub["arch"].map(lambda a: arch_order_map.get(a, 99))
        sub["_stage_rank"] = sub["stage"].map(lambda s: stage_order_map.get(s, 99))
        sub = sub.sort_values(["_arch_rank", "_stage_rank"]).reset_index(drop=True)

        max_val = sub[["s0_ms", "s1_ms"]].values.max()
        ylim = max_val * 1.40
        gap_s = max_val * 0.012  # gap for per-bar ms label
        gap_b = max_val * 0.055  # gap for speedup label above both bars

        x = np.arange(len(sub))
        for i, (_, row) in enumerate(sub.iterrows()):
            clr = ARCH_CLR.get(row["arch"], "#888")
            ax.bar(
                x[i] - w / 2, row["s0_ms"], width=w, color=clr, edgecolor="white", linewidth=0.5
            )
            ax.bar(
                x[i] + w / 2,
                row["s1_ms"],
                width=w,
                color=clr,
                hatch="////",
                edgecolor="white",
                linewidth=0.5,
            )
            # Small: absolute ms above each bar
            ax.text(
                x[i] - w / 2,
                row["s0_ms"] + gap_s,
                f"{row['s0_ms']:.1f}",
                ha="center",
                va="bottom",
                fontsize=6.5,
                color="#222",
            )
            ax.text(
                x[i] + w / 2,
                row["s1_ms"] + gap_s,
                f"{row['s1_ms']:.1f}",
                ha="center",
                va="bottom",
                fontsize=6.5,
                color="#222",
            )
            # Bigger: speedup centred above both bars
            ax.text(
                x[i],
                max(row["s0_ms"], row["s1_ms"]) + gap_b,
                f"{row['speedup']:.2f}×",
                ha="center",
                va="bottom",
                fontsize=10,
                fontweight="bold",
            )

        ax.set_ylim(0, ylim)
        ax.set_xticks(x)
        ax.set_xticklabels(
            [f"{ARCH_LABEL.get(r['arch'], r['arch'])}\n{r['stage']}" for _, r in sub.iterrows()],
            fontsize=9,
        )
        ax.set_ylabel("latency (ms)")
        ax.set_title(f"S0 vs S1 per-stage latency — {scen}")

        arch_handles = [
            Patch(facecolor=ARCH_CLR.get(a, "#888"), label=ARCH_LABEL.get(a, a))
            for a in ARCH_ORDER
            if a in sub["arch"].values
        ]
        style_handles = [
            Patch(facecolor="#888888", edgecolor="white", label="S0"),
            Patch(facecolor="#888888", hatch="////", edgecolor="white", label="S1"),
        ]
        ax.legend(
            handles=arch_handles + style_handles,
            fontsize=8,
            ncol=2,
            loc="upper left",
            framealpha=0.9,
        )

    fig.suptitle("S0 vs S1 per-stage latency (g_a / g_s)", fontsize=11)
    fig.tight_layout()
    savefig("s1_speedup")
    plt.show()

## §4 Throughput scaling curves

- **`nn_only`** — N concurrent DPU pipelines (`--dpu-cores N`). S1 baseline marks the channel-parallel target; N=1 equals s0 (implicit). S0 baseline toggleable via `SHOW_S0_BASELINE`.
- **`entropy_only`** — N concurrent entropy workers (`--entropy-threads N`). Measures EB(+GC for SH/ResSH) in isolation — no DPU, no I/O. Shown separately since it is not comparable to the full-pipeline s0/s1 fps. FP and SH annotated with speedup vs N=1.

In [ ]:
def _model_label(model_name: str) -> str:
    """Arch display name only (paper notation: SH, ResSH, FP, ResFP)."""
    m = _re.match(r"^([^-]+)-", model_name)
    arch = m.group(1) if m else model_name
    return ARCH_LABEL.get(arch, arch)


def _arch_rank(model_name: str) -> int:
    """Sort key: position in ARCH_ORDER (fast → slow)."""
    m = _re.match(r"^([^-]+)-", model_name)
    arch = m.group(1) if m else model_name
    return ARCH_ORDER.index(arch) if arch in ARCH_ORDER else 99


def plot_nn_only(show_s0_baseline: bool = True) -> None:
    """DPU parallelism scaling: throughput vs N concurrent DPU pipelines.

    show_s0_baseline: Toggle S0 baseline on/off (N=1 already lands on s0, so it is implicit).

    S1 baseline = channel-parallel target to beat.
    S0 baseline = single-pipeline reference (toggleable; implicit at N=1).
    """
    baseline_cfgs = [("s1", "-.")]
    if show_s0_baseline:
        baseline_cfgs.append(("s0", "--"))

    fig, ax = plt.subplots(figsize=(9, 6), dpi=FIG_DPI)
    model_handles = []

    for model_name in sorted(runs_df["model_name"].unique(), key=_arch_rank):
        arch = runs_df[runs_df["model_name"] == model_name]["arch"].iloc[0]
        clr = ARCH_CLR.get(arch, "#888888")

        sub = runs_df[
            (runs_df["model_name"] == model_name) & (runs_df["config"] == "nn_only")
        ].sort_values("dpu_cores")
        if not sub.empty:
            ax.plot(
                sub["dpu_cores"],
                sub["throughput_fps"],
                "o-",
                color=clr,
                linewidth=1.5,
                markersize=5,
            )
            model_handles.append(
                Line2D(
                    [0], [0], color=clr, marker="o", markersize=5, label=_model_label(model_name)
                )
            )

        for cfg, ls in baseline_cfgs:
            base = runs_df[
                (runs_df["model_name"] == model_name)
                & (runs_df["config"] == cfg)
                & (runs_df["scenario"] == "compress")
            ]
            if not base.empty:
                ax.axhline(
                    base["throughput_fps"].iloc[0],
                    color=clr,
                    linestyle=ls,
                    linewidth=0.9,
                    alpha=0.7,
                )

    extra_handles = [
        Line2D([0], [0], color="#555", linestyle="-.", linewidth=0.9, label="s1 baseline"),
    ]
    if show_s0_baseline:
        extra_handles.append(
            Line2D([0], [0], color="#555", linestyle="--", linewidth=0.9, label="s0 baseline")
        )

    ax.legend(handles=model_handles + extra_handles, fontsize=8, loc="upper left")
    ax.set_xlabel("DPU cores (N)")
    ax.set_ylim(bottom=0)
    ax.set_ylabel("throughput (fps)")
    ax.set_axisbelow(True)
    ax.grid(True, linewidth=0.4, alpha=0.35, color="#aaaaaa")
    ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))

    # Main title at figure level; subtitle just above axes via ax.set_title
    fig.suptitle("Throughput scaling — nn_only", fontsize=11, y=0.98)
    ax.set_title(
        "N=1 ≈ s0 (single pipeline); s1 = channel-parallel target to exceed",
        fontsize=7.5,
        style="italic",
    )
    plt.subplots_adjust(top=0.92)
    savefig("throughput_scaling_nn_only")
    plt.show()


plot_nn_only(show_s0_baseline=False)

In [ ]:
# Speedup annotations only for these two archs (ResFP/ResSH would overcrowd the plot)


def plot_entropy_only(entropy_annotated_arch: set[str] = {"FP", "SHyp"}) -> None:
    """Entropy-only scaling: throughput vs N concurrent entropy workers.

    Measures EB (FP/ResFP) or EB+GC (SH/ResSH) in isolation — no DPU, no I/O.
    FP and SH nodes at N=2,3,4 are annotated with their speedup vs N=1.
    """
    fig, ax = plt.subplots(figsize=(9, 6), dpi=FIG_DPI)
    model_handles = []

    for model_name in sorted(runs_df["model_name"].unique(), key=_arch_rank):
        arch = runs_df[runs_df["model_name"] == model_name]["arch"].iloc[0]
        clr = ARCH_CLR.get(arch, "#888888")

        sub = runs_df[
            (runs_df["model_name"] == model_name) & (runs_df["config"] == "entropy_only")
        ].sort_values("entropy_threads")
        if not sub.empty:
            ax.plot(
                sub["entropy_threads"],
                sub["throughput_fps"],
                "o-",
                color=clr,
                linewidth=1.5,
                markersize=5,
            )
            model_handles.append(
                Line2D(
                    [0], [0], color=clr, marker="o", markersize=5, label=_model_label(model_name)
                )
            )

            if arch in entropy_annotated_arch:
                base_rows = sub[sub["entropy_threads"] == 1]
                if not base_rows.empty:
                    fps_1 = base_rows["throughput_fps"].iloc[0]
                    for _, row in sub[sub["entropy_threads"] > 1].iterrows():
                        spd = row["throughput_fps"] / fps_1
                        ax.annotate(
                            f"{spd:.1f}×",
                            (row["entropy_threads"], row["throughput_fps"]),
                            textcoords="offset points",
                            xytext=(0, -20),
                            fontsize=9,
                            ha="center",
                            color=clr,
                        )

    ax.legend(handles=model_handles, fontsize=8, loc="upper left")
    ax.set_xlabel("entropy threads (N)")
    ax.set_ylim(bottom=0)
    ax.set_ylabel("throughput (fps)")
    ax.set_title("Throughput scaling — entropy_only")
    ax.set_axisbelow(True)
    ax.grid(True, linewidth=0.4, alpha=0.35, color="#aaaaaa")
    ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
    fig.tight_layout()
    savefig("throughput_scaling_entropy_only")
    plt.show()


plot_entropy_only(entropy_annotated_arch={"FP", "SHyp"})

## §5 Roofline

Theoretical DPU peak FPS per subgraph + static model info (workload OPs, INT8 weight / workspace / I/O byte sizes) — both extracted from `xdputil` at deploy time and stored in `results/benchmark_hardware/_roofline/<model>_xmodel_info.json`.

Section is skipped gracefully when the file is absent — run `collect_roofline.py` from the board after deploying a model to populate it.

```bash
# On the board (one-shot per model, ~60 s per DPU subgraph):
python3 /home/root/SAR_DDC/collect_roofline.py <model_name>
# Output: /home/root/SAR_DDC/bench_results/<model_name>_xmodel_info.json
# Then fetched host-side by scripts/fpga/benchmark/benchmark_sweep.py to results/benchmark_hardware/_roofline/
```


In [ ]:
ROOFLINE_DIR = BENCH_DIR / "_roofline"

# xmodel_info per model: {subgraphs: {func: {peak_fps, workload_ops, const_bytes, ...}}, totals}
xmodel_info = {}
for jf in ROOFLINE_DIR.glob("*_xmodel_info.json"):
    model_key = jf.stem.replace("_xmodel_info", "")
    with open(jf) as f:
        xmodel_info[model_key] = json.load(f)

if not xmodel_info:
    print(f"{y}No xmodel_info found in {ROOFLINE_DIR}{e}")
    print(f"{y}Run: python3 collect_roofline.py <model_name>  on the board, then re-sweep.{e}")
else:
    models_with_info = [m for m in runs_df["model_name"].unique() if m in xmodel_info]
    if not models_with_info:
        print(f"{y}xmodel_info files found but no model name matches runs_df.{e}")
    else:
        print(f"{g}xmodel_info loaded: {', '.join(models_with_info)}{e}")
        for m in models_with_info:
            info = xmodel_info[m]
            sgs = info["subgraphs"]
            print(f"\n  {m}")
            print(f"    total_workload_ops = {info['total_workload_ops']:>15,}")
            print(f"    total_const_bytes  = {info['total_const_bytes']:>15,}")
            for sg, d in sgs.items():
                print(
                    f"    {sg:4s}  peak={d['peak_fps']:7.2f} fps  "
                    f"ops={d['workload_ops']:>12,}  "
                    f"const={d['const_bytes']:>10,} B  "
                    f"workspace={d['workspace_bytes']:>10,} B"
                )

# Backward-compatible roof_data: flat {model: {func: peak_fps}} for the existing §5 plot
roof_data = {
    m: {func: d["peak_fps"] for func, d in info["subgraphs"].items()}
    for m, info in xmodel_info.items()
}


In [ ]:
ROOFLINE_CLR = CFG_CLR["roofline"]


def plot_roofline_bars(annotate_eff=True):
    all_models = [
        m
        for arch in ARCH_ORDER
        for m in runs_df["model_name"].unique()
        if runs_df[runs_df["model_name"] == m]["arch"].iloc[0] == arch and m in roof_data
    ]
    if not all_models:
        print(f"{y}No roofline data available.{e}")
        return

    fp_family = [
        m
        for m in all_models
        if runs_df[runs_df["model_name"] == m]["arch"].iloc[0] in ("FP", "SHyp")
    ]
    res_family = [
        m
        for m in all_models
        if runs_df[runs_df["model_name"] == m]["arch"].iloc[0] in ("ResFP", "ResSHyp")
    ]
    panels = [(fp_family, "FP / SH"), (res_family, "ResFP / ResSH")]

    fig, axes = plt.subplots(1, 2, figsize=(13, 5.8), dpi=FIG_DPI)
    fig.subplots_adjust(bottom=0.20, wspace=0.16, top=0.82, left=0.07, right=0.97)

    fig.suptitle(
        "DPU roofline: achieved throughput vs subgraph ceilings  (compress, s0 / s1)",
        fontsize=11,
        y=0.97,
    )
    fig.text(
        0.5,
        0.90,
        "peaks from xdputil benchmark (one subgraph in isolation, zero host overhead)  ·  "
        "DPU pipeline ceil = 1/Σ(1/peak_i), all DPU subgraphs sequential  ·  "
        "h_a / h_s >1200 fps — included in DPU ceil, off-scale here",
        ha="center",
        va="top",
        fontsize=6.5,
        color="#555",
    )

    sg_styles = {"g_a": ("--", 1.4), "g_s": (":", 1.8)}
    legend_sg = {}

    def draw_panel(ax, models, panel_title):
        w = 0.28
        span = 0.44  # half-width of DPU ceil partial segments

        scale_ceilings = [
            roof_data[m][sg]
            for m in models
            for sg in ("g_a", "g_s")
            if sg in roof_data[m] and isinstance(roof_data[m][sg], (int, float))
        ]
        y_top = max(scale_ceilings) * 1.30 if scale_ceilings else 50
        label_gap = y_top * 0.030
        label_h = y_top * 0.058

        ax.set_axisbelow(True)
        ax.grid(True, which="major", linewidth=0.4, alpha=0.3, color="#aaaaaa")
        ax.set_ylim(0, y_top)

        # ── bars + combined fps (eff%) annotation ──────────────────────────────
        for idx, model in enumerate(models):
            arch = runs_df[runs_df["model_name"] == model]["arch"].iloc[0]
            clr = ARCH_CLR.get(arch, "#888")
            peaks = roof_data[model]
            x = float(idx)

            all_sg = [
                sg
                for sg in ("g_a", "g_s", "h_a", "h_s")
                if sg in peaks and isinstance(peaks[sg], (int, float))
            ]
            dpu_ceil = 1 / sum(1 / peaks[sg] for sg in all_sg) if all_sg else None

            for cfg, xi, hatch in [("s0", x - w / 2 - 0.02, ""), ("s1", x + w / 2 + 0.02, "////")]:
                row = runs_df[
                    (runs_df["model_name"] == model)
                    & (runs_df["config"] == cfg)
                    & (runs_df["scenario"] == "compress")
                ]
                if row.empty:
                    continue
                fps = row["throughput_fps"].iloc[0]
                ax.bar(
                    xi,
                    fps,
                    width=w,
                    color=clr,
                    hatch=hatch,
                    edgecolor="white",
                    linewidth=0.6,
                    zorder=4,
                )
                if annotate_eff and dpu_ceil:
                    eff = fps / dpu_ceil * 100
                    lbl = f"{fps:.1f} ({eff:.0f}%)"
                else:
                    lbl = f"{fps:.1f}"
                ax.text(
                    xi,
                    fps + label_gap * 0.45,
                    lbl,
                    ha="center",
                    va="bottom",
                    fontsize=8,
                    color="black",
                    zorder=5,
                )

        # ── full-width lines for g_a / g_s ─ centered labels, bumped if close ──
        sg_panel = []
        for sg, (ls, lw) in sg_styles.items():
            vals = [
                roof_data[m][sg]
                for m in models
                if sg in roof_data[m] and isinstance(roof_data[m][sg], (int, float))
            ]
            if vals:
                sg_panel.append((sum(vals) / len(vals), sg, ls, lw))
                legend_sg[sg] = (ls, lw)
        sg_panel.sort(key=lambda t: t[0])

        cx = (len(models) - 1) / 2.0
        next_lbl_y = -999
        for fps_avg, sg, ls, lw in sg_panel:
            ax.axhline(
                fps_avg, color=ROOFLINE_CLR, linestyle=ls, linewidth=lw, alpha=0.72, zorder=2
            )
            y_lbl = max(fps_avg + label_gap * 0.5, next_lbl_y)
            ax.text(
                cx,
                y_lbl,
                f"{sg}  {fps_avg:.0f}",
                ha="center",
                va="bottom",
                fontsize=7.5,
                color="black",
                zorder=5,
            )
            next_lbl_y = y_lbl + label_h

        # ── partial lines for DPU ceil ─ one per arch, fps label left + above ──
        for idx, model in enumerate(models):
            peaks = roof_data[model]
            x = float(idx)

            all_sg = [
                sg
                for sg in ("g_a", "g_s", "h_a", "h_s")
                if sg in peaks and isinstance(peaks[sg], (int, float))
            ]
            if not all_sg:
                continue
            dpu_ceil = 1 / sum(1 / peaks[sg] for sg in all_sg)

            ax.plot(
                [x - span, x + span],
                [dpu_ceil, dpu_ceil],
                color=ROOFLINE_CLR,
                linestyle="-",
                linewidth=2.2,
                alpha=0.92,
                zorder=3,
            )
            ax.text(
                x - span,
                dpu_ceil + label_gap * 0.55,
                f"{dpu_ceil:.0f}",
                ha="right",
                va="bottom",
                fontsize=7.5,
                color="black",
                fontweight="bold",
                zorder=5,
            )

        ax.set_xticks(range(len(models)))
        ax.set_xticklabels(
            [
                ARCH_LABEL.get(runs_df[runs_df["model_name"] == m]["arch"].iloc[0], m)
                for m in models
            ],
            fontsize=12,
        )
        ax.set_ylabel("throughput (fps)", fontsize=10)
        ax.set_xlim(-0.65, (len(models) - 1) + 0.65)
        ax.set_title(panel_title, fontsize=11, pad=3)

    for ax, (models, title) in zip(axes, panels):
        draw_panel(ax, models, title)

    # ── shared legend ─────────────────────────────────────────────────────────
    arch_h = [
        Patch(facecolor=ARCH_CLR.get(a, "#888"), label=ARCH_LABEL.get(a, a))
        for a in ARCH_ORDER
        if any(runs_df[runs_df["model_name"] == m]["arch"].iloc[0] == a for m in all_models)
    ]
    style_h = [
        Patch(facecolor="#777", edgecolor="white", label="s0"),
        Patch(facecolor="#777", hatch="////", edgecolor="white", label="s1"),
    ]
    ceil_h = [
        Line2D([0], [0], color=ROOFLINE_CLR, linestyle=ls, linewidth=lw, label=sg)
        for sg, (ls, lw) in legend_sg.items()
    ] + [
        Line2D(
            [0], [0], color=ROOFLINE_CLR, linestyle="-", linewidth=2.2, label="DPU pipeline ceil"
        )
    ]
    fig.legend(
        handles=arch_h + style_h + ceil_h,
        loc="lower center",
        ncol=5,
        fontsize=9,
        framealpha=0.9,
        bbox_to_anchor=(0.5, 0.03),
    )
    savefig("roofline_bars")
    plt.show()


plot_roofline_bars()


### §5b Williams roofline (per-subgraph)

Each DPU subgraph is one point: **x** = arithmetic intensity (`workload_ops ÷ bytes crossing DDR`), **y** = achievable throughput (`workload_ops × single-thread peak fps`). The solid line is the combined roofline = `min(memory-bandwidth slope, compute-peak flat)`.

- Points **left of the knee** are memory-bandwidth bound; points **on the flat** are compute bound.
- λ-**invariant**: workload, bytes and peak fps don't depend on λ (N=128, M=256 fixed), so this figure is identical for every λ of a given arch.
- `workload` counts **ops** (Xilinx: 1 MAC = 2 ops); DPU peak = 1.23 TOPS. Memory ceiling uses DDR **peak** 19.2 GB/s — true sustained is ~60–80%, so the slope is optimistic.


In [ ]:
# ── §5b Williams roofline (per-subgraph) ──────────────────────────────────────
# Ceilings (see project assumptions):
#   DPU compute peak = B4096 @ 300 MHz = 4096 ops/clk × 300e6 = 1.2288 TOPS
#     (Xilinx counts 1 MAC = 2 ops; xdputil "workload" is in ops, so this matches).
#   DDR bandwidth    = 19.2 GB/s DDR4-2400 PEAK; real sustained is ~60-80% of this.
DPU_PEAK_OPS_S = 1.2288e12
DDR_BW_BYTES_S = 19.2e9
SG_MARKER = {"g_a": "o", "g_s": "s", "h_a": "^", "h_s": "D"}


def plot_roofline_williams():
    if not xmodel_info:
        print(f"{y}No xmodel_info — run §5 loader first.{e}")
        return

    # FP↔SH share g_a/g_s; SH↔ResSH share h_a/h_s. Per-arch x-jitter surfaces all.
    JITTER = {"FP": 0.92, "SHyp": 0.97, "ResFP": 1.03, "ResSHyp": 1.08}
    points = []  # (arch, sg, arithmetic_intensity, ops_per_s)
    for model, d in xmodel_info.items():
        arch = model.split("-", 1)[0]
        for sg, m in d["subgraphs"].items():
            tb = m["const_bytes"] + m["workspace_bytes"] + m["input_bytes"] + m["output_bytes"]
            ai = m["workload_ops"] / tb if tb else 0
            points.append((arch, sg, ai, m["workload_ops"] * m["peak_fps"]))

    fig, ax = plt.subplots(figsize=(9, 6.5), dpi=FIG_DPI)
    ais = [p[2] for p in points]
    x_min, x_max = min(ais) * 0.4, max(ais) * 2.5
    xs = np.geomspace(x_min, x_max, 200)
    ax.plot(
        xs, np.minimum(xs * DDR_BW_BYTES_S, DPU_PEAK_OPS_S), color="#222", linewidth=2.0, zorder=2
    )

    knee = DPU_PEAK_OPS_S / DDR_BW_BYTES_S
    ax.text(
        xs[-1] * 0.92,
        DPU_PEAK_OPS_S * 0.78,
        f"compute  {DPU_PEAK_OPS_S / 1e12:.2f} TOPS",
        ha="right",
        va="top",
        fontsize=9,
        color="#222",
        style="italic",
    )
    ax.text(
        knee * 0.30,
        knee * 0.30 * DDR_BW_BYTES_S * 2.5,
        f"memory  {DDR_BW_BYTES_S / 1e9:.1f} GB/s (peak)",
        ha="center",
        va="bottom",
        fontsize=9,
        color="#222",
        style="italic",
    )

    plotted_a, plotted_s = set(), set()
    for arch, sg, ai, ops_s in points:
        ax.scatter(
            ai * JITTER.get(arch, 1.0),
            ops_s,
            color=ARCH_CLR.get(arch, "#888"),
            marker=SG_MARKER.get(sg, "x"),
            s=120,
            zorder=4,
            edgecolor="white",
            linewidth=0.9,
            alpha=0.92,
        )
        plotted_a.add(arch)
        plotted_s.add(sg)

    # One % label per (subgraph, AI-cluster) — utilisation vs the local ceiling
    clusters = {}
    for arch, sg, ai, ops_s in points:
        clusters.setdefault((sg, round(np.log10(ai) * 4) / 4), []).append((ai, ops_s))
    for (sg, _), pts in clusters.items():
        ai_a = np.mean([p[0] for p in pts])
        ops_a = np.mean([p[1] for p in pts])
        ceil = min(ai_a * DDR_BW_BYTES_S, DPU_PEAK_OPS_S)
        ax.annotate(
            f"{ops_a / ceil * 100:.0f}%",
            (ai_a, ops_a),
            textcoords="offset points",
            xytext=(0, 8),
            fontsize=8,
            color="#222",
            fontweight="bold",
        )

    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel("Arithmetic intensity (ops / byte)")
    ax.set_ylabel("Achievable throughput (ops / s)")
    ax.set_title("DPU Williams roofline — per-subgraph compute vs memory bound")
    ax.grid(True, which="both", linewidth=0.3, alpha=0.4)
    ax.set_xlim(x_min, x_max)
    ax.set_ylim(min(p[3] for p in points) * 0.55, DPU_PEAK_OPS_S * 2.0)

    arch_h = [
        Patch(facecolor=ARCH_CLR.get(a, "#888"), label=ARCH_LABEL.get(a, a))
        for a in ARCH_ORDER
        if a in plotted_a
    ]
    sg_h = [
        Line2D(
            [0], [0], marker=SG_MARKER[s], color="#444", markersize=9, linestyle="None", label=s
        )
        for s in ("g_a", "g_s", "h_a", "h_s")
        if s in plotted_s
    ]
    leg1 = ax.legend(handles=arch_h, loc="lower right", fontsize=9, title="arch", framealpha=0.9)
    ax.add_artist(leg1)
    ax.legend(
        handles=sg_h,
        loc="lower right",
        bbox_to_anchor=(0.78, 0.0),
        fontsize=9,
        title="subgraph",
        framealpha=0.9,
    )
    fig.text(
        0.5,
        0.01,
        "workload = ops (xdputil; 1 MAC = 2 ops)  ·  % = achieved / min(memory, compute)  ·  "
        "memory ceiling is DDR peak — sustained ~60-80%  ·  per-arch x-jitter",
        ha="center",
        fontsize=7,
        color="#666",
    )
    fig.tight_layout(rect=(0.0, 0.03, 1.0, 1.0))
    savefig("roofline_williams")
    plt.show()


plot_roofline_williams()


## §6 Power & energy

Requires runs with a `power` block in the JSON (produced when `benchmark_hardware` is run with
a power-sampling thread).  Gracefully skipped when no power data is available.

**Power terminology** (from `main_benchmark.cpp`: "idle baseline before, active window around benchmark"):
- **idle** (`idle_*_W`) — power during a quiet ~10 s baseline window, no inference. The always-on cost of the board (MPSoC, PL fabric, peripherals).
- **active** (`active_*_W`) — power *during* the inference run. This is the total power **under load** — it **includes** the idle baseline, it is not a delta.
- **dynamic** = `active − idle` — the marginal power attributable to the workload. On this board `active ≈ idle` (~9.3 vs ~8.6 W MPSoC), so the dynamic term is a small difference of two large numbers and is **noise-sensitive** — prefer reasoning with gross `active`, or isolate the `DPU_fabric` rail.

`energy_mJ_per_patch = active_MPSoC_W × total_latency_mean_ms` uses **gross** active power.


In [ ]:
power_df = runs_df.dropna(subset=["active_MPSoC_W"]).copy()

if power_df.empty:
    print(f"{y}No power data — re-run benchmark_hardware with power sampling enabled.{e}")
    idle_MPSoC_mean_W = None
else:
    # Average idle power across all instrumented runs — used as reference line below
    idle_MPSoC_mean_W = power_df["idle_MPSoC_W"].mean()
    print(
        f"{g}{len(power_df)} runs with power data.  "
        f"idle_MPSoC_mean_W = {idle_MPSoC_mean_W:.2f} W{e}"
    )


def plot_power_energy(configs=("s0", "s1"), scenario="compress"):
    if power_df.empty:
        print(f"{y}No power data.{e}")
        return

    pf = power_df[power_df["config"].isin(configs) & (power_df["scenario"] == scenario)].copy()
    if pf.empty:
        print(f"{y}No rows for configs={configs} scenario={scenario}{e}")
        return

    # One group per arch (in canonical order), one bar per config inside the group
    archs = [a for a in ARCH_ORDER if a in pf["arch"].values]
    x = np.arange(len(archs))
    w = 0.30
    cfg_offsets = {"s0": -w / 2 - 0.02, "s1": w / 2 + 0.02}
    cfg_hatches = {"s0": "", "s1": "////"}

    fig, axes = plt.subplots(1, 2, figsize=(13, 5.2), dpi=FIG_DPI)

    for ax in axes:
        ax.set_axisbelow(True)
        ax.yaxis.grid(True, linewidth=0.4, alpha=0.35, color="#aaaaaa")
        ax.set_xticks(x)
        ax.set_xticklabels([ARCH_LABEL.get(a, a) for a in archs], fontsize=11)

    # ── active power + idle reference line ────────────────────────────────────
    ax = axes[0]
    for ai, arch in enumerate(archs):
        clr = ARCH_CLR.get(arch, "#888")
        for cfg in configs:
            row = pf[(pf["arch"] == arch) & (pf["config"] == cfg)]
            if row.empty:
                continue
            ax.bar(
                x[ai] + cfg_offsets.get(cfg, 0),
                row["active_MPSoC_W"].iloc[0],
                width=w,
                color=clr,
                hatch=cfg_hatches.get(cfg, ""),
                edgecolor="white",
                linewidth=0.6,
                zorder=3,
            )
    if idle_MPSoC_mean_W is not None:
        ax.axhline(
            idle_MPSoC_mean_W,
            color="#444",
            linestyle="--",
            linewidth=1.3,
            zorder=2,
            label=f"idle avg  {idle_MPSoC_mean_W:.1f} W",
        )
        ax.legend(fontsize=8, loc="upper left")
    ax.set_ylim(bottom=0)
    ax.set_ylabel("power (W)")
    ax.set_title(f"Active MPSoC power  ({scenario})")

    # ── energy per patch ──────────────────────────────────────────────────────
    ax = axes[1]
    y_max = pf["energy_mJ_per_patch"].max()
    for ai, arch in enumerate(archs):
        clr = ARCH_CLR.get(arch, "#888")
        for cfg in configs:
            row = pf[(pf["arch"] == arch) & (pf["config"] == cfg)]
            if row.empty:
                continue
            val = row["energy_mJ_per_patch"].iloc[0]
            xi = x[ai] + cfg_offsets.get(cfg, 0)
            ax.bar(
                xi,
                val,
                width=w,
                color=clr,
                hatch=cfg_hatches.get(cfg, ""),
                edgecolor="white",
                linewidth=0.6,
                zorder=3,
            )
            ax.text(xi, val + y_max * 0.02, f"{val:.0f}", ha="center", va="bottom", fontsize=8)
    ax.set_ylim(bottom=0)
    ax.set_ylabel("energy (mJ / patch)")
    ax.set_title("Energy per patch  (active_MPSoC_W × latency_ms)")

    # ── shared legend under both plots ────────────────────────────────────────
    arch_h = [Patch(facecolor=ARCH_CLR.get(a, "#888"), label=ARCH_LABEL.get(a, a)) for a in archs]
    style_h = [
        Patch(facecolor="#888", edgecolor="white", label="s0"),
        Patch(facecolor="#888", hatch="////", edgecolor="white", label="s1"),
    ]
    fig.legend(
        handles=arch_h + style_h,
        loc="lower center",
        ncol=len(arch_h) + len(style_h),
        fontsize=9,
        framealpha=0.9,
        bbox_to_anchor=(0.5, 0.0),
    )
    fig.suptitle(f"Power & energy  ({scenario})", fontsize=11)
    fig.tight_layout(rect=(0.0, 0.05, 1.0, 0.97))
    savefig("power_energy")
    plt.show()


plot_power_energy()


### §6b Energy per DPU-op

Normalises whole-system energy by DPU work: `pJ/op = MPSoC power × latency × 1e9 ÷ total_workload_ops`. This disentangles *"this arch is wasteful"* from *"this arch does more work"* — the small FP/SH models look cheap in mJ/patch but pay the fixed idle+CPU overhead over far fewer ops.

**Caveat**: the numerator is whole-system energy (DPU + CPU entropy + idle baseline), so this is *system energy per DPU-op*, **not** isolated DPU efficiency. `net_of_idle=True` subtracts the idle baseline but is noise-sensitive (active ≈ idle on this board), so **gross is the default**. A clean DPU-only metric would need the `DPU_fabric` rail + a DPU-only time window (future work).


In [ ]:
# ── §6b Energy per DPU-op ─────────────────────────────────────────────────────
def plot_energy_per_op(net_of_idle=False):
    """pJ per DPU-op = MPSoC power × latency × 1e9 / total_workload_ops.

    net_of_idle=False (default): gross active power — robust.
    net_of_idle=True: active − idle (dynamic) — noise-sensitive (active≈idle), use with care.
    """
    if power_df.empty:
        print(f"{y}No power data.{e}")
        return
    pf = power_df[
        power_df["config"].isin(("s0", "s1")) & (power_df["scenario"] == "compress")
    ].copy()

    def pj(row):
        if row["model_name"] not in xmodel_info:
            return np.nan
        total = xmodel_info[row["model_name"]]["total_workload_ops"]
        power = row["active_MPSoC_W"]
        if net_of_idle and not pd.isna(row.get("idle_MPSoC_W")):
            power = row["active_MPSoC_W"] - row["idle_MPSoC_W"]
        return power * row["total_latency_mean_ms"] * 1e9 / total

    pf = pf.assign(pJ_per_op=pf.apply(pj, axis=1)).dropna(subset=["pJ_per_op"])
    if pf.empty:
        print(f"{y}No power+xmodel_info overlap.{e}")
        return

    archs = [a for a in ARCH_ORDER if a in pf["arch"].values]
    x = np.arange(len(archs))
    w = 0.30
    off = {"s0": -w / 2 - 0.02, "s1": w / 2 + 0.02}
    hatch = {"s0": "", "s1": "////"}

    fig, ax = plt.subplots(figsize=(9, 5.5), dpi=FIG_DPI)
    ax.set_axisbelow(True)
    ax.yaxis.grid(True, linewidth=0.4, alpha=0.35, color="#aaaaaa")
    ymax = pf["pJ_per_op"].max()
    for ai, arch in enumerate(archs):
        for cfg in ("s0", "s1"):
            row = pf[(pf["arch"] == arch) & (pf["config"] == cfg)]
            if row.empty:
                continue
            val = row["pJ_per_op"].iloc[0]
            xi = x[ai] + off[cfg]
            ax.bar(
                xi,
                val,
                width=w,
                color=ARCH_CLR.get(arch, "#888"),
                hatch=hatch[cfg],
                edgecolor="white",
                linewidth=0.6,
                zorder=3,
            )
            ax.text(xi, val + ymax * 0.02, f"{val:.2f}", ha="center", va="bottom", fontsize=8)
    ax.set_xticks(x)
    ax.set_xticklabels([ARCH_LABEL.get(a, a) for a in archs], fontsize=11)
    ax.set_ylabel("pJ / op")
    ax.set_ylim(bottom=0)
    kind = "net-of-idle" if net_of_idle else "gross (incl. idle)"
    ax.set_title(f"Whole-system energy per DPU-op  ({kind}, compress)")
    arch_h = [Patch(facecolor=ARCH_CLR.get(a, "#888"), label=ARCH_LABEL.get(a, a)) for a in archs]
    style_h = [
        Patch(facecolor="#888", edgecolor="white", label="s0"),
        Patch(facecolor="#888", hatch="////", edgecolor="white", label="s1"),
    ]
    ax.legend(handles=arch_h + style_h, fontsize=9, loc="upper right", framealpha=0.9)
    fig.text(
        0.5,
        0.01,
        "pJ/op = MPSoC power × latency × 1e9 / total_workload_ops.  Whole-system energy "
        "(DPU + CPU entropy + idle) per DPU-op — NOT isolated DPU efficiency.",
        ha="center",
        fontsize=7,
        color="#666",
    )
    fig.tight_layout(rect=(0.0, 0.03, 1.0, 1.0))
    savefig("energy_per_op")
    plt.show()


plot_energy_per_op()


## §7 Triple tradeoff

The project optimises **three competing objectives simultaneously**:

| Axis | Metric | Who wins |
| --- | --- | --- |
| Compression rate | bpp ↓ | lower λ models |
| Image quality | PSNR / SSIM ↑ | higher λ, ResSHyp |
| HW efficiency | throughput_fps ↑ | FP, S1 |

Three scatter plots make the tradeoff visible.  
Requires `join_hw_quality` to have at least one benchmarked model with quality data.

In [ ]:
# Use S0 compress rows for a fair apples-to-apples comparison across architectures
trade_df = joined_df[(joined_df["config"] == "s0") & (joined_df["scenario"] == "compress")].copy()

if trade_df["bpp"].isna().all():
    print(
        f"{y}No quality data joined — benchmark ResSHyp-relu_s1_L1000_pt and FP-relu_s1_L1000_pt{e}"
    )
    print(f"{y}(seed-1 versions have metrics.json in compiled_models/) then re-run §0.{e}")
else:
    fig, axes = plt.subplots(1, 3, figsize=(21, 6), dpi=FIG_DPI)

    def scatter(ax, x_col, y_col, size_col=None, xlabel="", ylabel="", title=""):
        for _, row in trade_df.iterrows():
            if pd.isna(row.get(x_col)) or pd.isna(row.get(y_col)):
                continue
            clr = ARCH_CLR.get(row["arch"], "#888")
            sz = 120 if size_col is None else max(20, row.get(size_col, 1) * 60)
            ax.scatter(
                row[x_col],
                row[y_col],
                color=clr,
                s=sz,
                zorder=3,
                edgecolors="white",
                linewidths=0.5,
            )
            ax.annotate(
                row["arch"],
                (row[x_col], row[y_col]),
                textcoords="offset points",
                xytext=(4, 4),
                fontsize=7,
            )
        ax.set_xlabel(xlabel)
        ax.set_ylabel(ylabel)
        ax.set_title(title)
        ax.grid(True, linewidth=0.4, alpha=0.5)

    scatter(
        axes[0],
        "bpp",
        "throughput_fps",
        xlabel="bpp ↓",
        ylabel="throughput (fps) ↑",
        title="Compression rate vs HW efficiency",
    )

    scatter(
        axes[1],
        "bpp",
        "energy_mJ_per_patch",
        xlabel="bpp ↓",
        ylabel="energy (mJ/patch) ↓",
        title="Compression rate vs energy",
    )

    scatter(
        axes[2],
        "psnr_MERLIN",
        "throughput_fps",
        size_col="bpp",
        xlabel="PSNR MERLIN (dB) ↑",
        ylabel="throughput (fps) ↑",
        title="Quality vs HW efficiency (point size ∝ bpp)",
    )

    # Legend: architecture colours
    from matplotlib.patches import Patch

    handles = [
        Patch(facecolor=clr, label=arch)
        for arch, clr in ARCH_CLR.items()
        if not arch.startswith("_")
    ]
    fig.legend(
        handles=handles,
        loc="upper center",
        ncol=len(handles),
        bbox_to_anchor=(0.5, 1.03),
        fontsize=9,
    )

    fig.tight_layout()
    savefig("triple_tradeoff")
    plt.show()

### §7b Cost vs quality (design-space)

Compute cost (`total_workload_ops`/patch — λ-invariant per arch) vs **FPGA-eval** PSNR (`metrics.json`). Dashed lines connect each arch to its residual variant, annotated with the dB gained per × of extra compute — answers *"does the extra compute pay for itself?"*

**Currently single-seed.** Seed-averaging (mean ± std over the 6 compiled seeds) is a planned upgrade — see §8.


In [ ]:
# ── §7b Cost vs quality (design-space) ────────────────────────────────────────
def plot_cost_vs_quality():
    """Compute cost (total ops/patch, λ-invariant per arch) vs FPGA-eval PSNR.

    Single seed for now — seed-averaging is a planned upgrade (see §8).
    """
    if quality_df.empty:
        print(f"{y}No quality data.{e}")
        return
    qf = quality_df[quality_df["model_name"].isin(xmodel_info)].copy()
    if qf.empty:
        print(f"{y}No quality rows match benchmarked models {set(xmodel_info)}.{e}")
        return
    qf["total_workload_ops"] = qf["model_name"].map(lambda m: xmodel_info[m]["total_workload_ops"])
    qf["arch"] = qf["model_name"].map(lambda m: m.split("-", 1)[0])

    fig, ax = plt.subplots(figsize=(9, 6), dpi=FIG_DPI)
    ax.set_axisbelow(True)
    ax.grid(True, linewidth=0.4, alpha=0.35, color="#aaaaaa")
    bmin, bmax = qf["bpp"].min(), qf["bpp"].max()

    def s_of(b):
        return 160 if (pd.isna(b) or bmax == bmin) else 80 + (b - bmin) / (bmax - bmin) * 260

    # Connect each arch to its residual variant: shows dB gained per × of compute
    for base, ext in {"FP": "ResFP", "SHyp": "ResSHyp"}.items():
        br = qf[qf["arch"] == base]
        er = qf[qf["arch"] == ext]
        if br.empty or er.empty:
            continue
        x0, y0 = br["total_workload_ops"].iloc[0], br["psnr_MERLIN"].iloc[0]
        x1, y1 = er["total_workload_ops"].iloc[0], er["psnr_MERLIN"].iloc[0]
        ax.plot(
            [x0, x1], [y0, y1], color="#888", linestyle="--", linewidth=1.0, alpha=0.6, zorder=1
        )
        ax.text(
            (x0 + x1) / 2,
            (y0 + y1) / 2 + 0.05,
            f"+{y1 - y0:.2f} dB  @  {x1 / x0:.1f}× ops",
            ha="center",
            va="bottom",
            fontsize=8,
            color="#444",
            fontstyle="italic",
        )

    for _, r in qf.iterrows():
        ax.scatter(
            r["total_workload_ops"],
            r["psnr_MERLIN"],
            s=s_of(r["bpp"]),
            color=ARCH_CLR.get(r["arch"], "#888"),
            edgecolor="white",
            linewidth=0.9,
            zorder=3,
            alpha=0.92,
        )
        side = 1 if r["arch"] in ("FP", "SHyp") else -1
        ax.annotate(
            f"{ARCH_LABEL.get(r['arch'], r['arch'])}  bpp={r['bpp']:.2f}",
            (r["total_workload_ops"], r["psnr_MERLIN"]),
            textcoords="offset points",
            xytext=(12 * side, -2),
            fontsize=9,
            color="#222",
            va="center",
            ha="left" if side > 0 else "right",
        )

    ax.set_xlim(0, qf["total_workload_ops"].max() * 1.20)
    ax.set_xlabel("Compute cost  (total ops / patch)")
    ax.set_ylabel("Quality  (PSNR MERLIN, dB)")
    ax.set_title("Cost vs quality — does the extra compute pay for itself?")
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda t, _: f"{t / 1e9:.0f} G"))
    arch_h = [
        Patch(facecolor=ARCH_CLR.get(a, "#888"), label=ARCH_LABEL.get(a, a))
        for a in ARCH_ORDER
        if a in qf["arch"].values
    ]
    ax.legend(handles=arch_h, loc="lower right", fontsize=9, title="arch", framealpha=0.9)
    fig.text(
        0.5,
        0.01,
        "FPGA-eval metrics (metrics.json), SINGLE seed.  Point size ∝ bpp.  "
        "Dashed line = non-residual ↔ residual within an arch family.",
        ha="center",
        fontsize=7,
        color="#666",
    )
    fig.tight_layout(rect=(0.0, 0.03, 1.0, 1.0))
    savefig("cost_vs_quality")
    plt.show()


plot_cost_vs_quality()


## §8 Planned upgrades (not yet built)

Captured so future-me can pick up without re-deriving. Design detail in the project memory `project_roofline_model_info`.

1. **§7b → seed-averaged** — switch cost-vs-quality (and any RD plot) from single-seed to **mean ± std over the 6 seeds** per (arch, λ). `quality_df` already parses `seed`; aggregate by `(arch, lambda)`. FPGA-eval metrics only. Zero new board runs.
2. **Whole-model roofline** (companion to §5b) — one point per (arch, config): x = whole-model ops/byte, y = `total_ops × measured_throughput_fps` (real pipeline fps from `runs_df`, includes entropy/normalize/data-movement). The gap below the §5b per-subgraph points = the **host-overhead tax**. High insight/effort ratio.
3. **RD curve annotated by compute cost** — *separate new notebook*. Classic bpp×PSNR RD curve per arch (cf. `RD-curve_ablation.ipynb`), with compute cost encoded by colour-gradient **or** side-annotation (decide then). Seed-averaged FPGA metrics. Zero board runs.
4. **DPU-only energy** — replace whole-system pJ/op with `DPU_fabric` rail power + a DPU-only time window for true subgraph energy efficiency. Needs new instrumentation; lower priority.
5. **APM DDR-traffic verification** — replace the `const+workspace+in+out` arithmetic-intensity estimate with measured DDR bytes from the AXI Performance Monitor during single-subgraph runs. Only if memory-bound classification needs hardening.
